# SQL Analysis

Objetivo: responder las preguntas de negocio clave consultando directamente las tablas Gold con Spark SQL.

Pregunta 1: ¿Cuál es la tendencia temporal del mercado?   
Pregunta 2: ¿Qué ciudades tienen mayor volumen y cómo se comparan sus precios?   
Pregunta 3: ¿Qué tipo de vivienda se vende más en Londres?    
Pregunta 4: ¿Cómo se distribuye el mercado londinense por categoría de precio?   
Pregunta 5: ¿Cómo varían los precios entre distritos londinenses?   

In [0]:
-- ============================================
-- SQL ANALYSIS & VISUALIZATION
-- ============================================
-- Análisis de transacciones inmobiliarias UK
-- Queries SQL sobre tablas Gold
-- ============================================

SELECT ' Notebook 3: SQL Analysis iniciado' as status;

In [0]:
-- Ver todas las tablas disponibles
SHOW TABLES IN workspace.uk_housing;

## Análisis general UK

### Query 1 — Tendencia temporal
Precio medio mensual, volatilidad (std) y valor total del mercado a nivel nacional. Fuente: `gold_temporal_trends`.

In [0]:
-- ============================================
-- Analizar la tendencia del mercado inmobiliario en cuanto a precios
-- ============================================
SELECT 
    year,
    month,
    SUM(total_transactions) AS total_transactions,
    ROUND(AVG(avg_price), 0) AS avg_price,
    ROUND(AVG(std_price), 0) AS price_volatility,
    ROUND(SUM(total_value), 0) AS total_market_value
FROM workspace.uk_housing.gold_temporal_trends
GROUP BY year, month
ORDER BY year, month;

### Query 2 — Ciudades por volumen
Top 10 ciudades ordenadas por número de transacciones. Incluye precio medio, mínimo y máximo.

In [0]:
-- ============================================
-- Identificar qué ciudades tienen mayor volumen de transacciones y comparar precios
-- ============================================

SELECT 
    town_city,
    total_transactions,
    ROUND(avg_price, 0) AS avg_price,
    ROUND(min_price, 0) AS min_price,
    ROUND(max_price, 0) AS max_price
FROM workspace.uk_housing.gold_city_prices
ORDER BY total_transactions DESC
LIMIT 10;

## Análisis de Londres (la ciudad con más transacciones)

### Query 3 — Tipos de propiedad en Londres
Análisis centrado en Londres, la ciudad con más transacciones. Muestra volumen y precio por tipología.

In [0]:
-- ============================================
-- Analizar qué tipo de vivienda se vende más y sus precios
-- ============================================
SELECT 
    property_type_desc,
    total_transactions,
    ROUND(total_price, 0) AS total_market_value,
    ROUND(avg_price, 0) AS avg_price
FROM workspace.uk_housing.gold_property_analysis
WHERE town_city = 'LONDON'
ORDER BY total_transactions DESC;

### Query 4 — Segmentos de precio en Londres
Distribución de las transacciones londinenses en los 4 segmentos de precio, con cuota de mercado (window function).

In [0]:
-- ============================================
-- Analizar y entender la distribución del mercado inmobiliario por categorías de precios en Londres
-- ============================================
SELECT 
    price_category,
    total_transactions,
    ROUND(avg_price, 0) AS avg_price,
    ROUND(min_price, 0) AS min_price,
    ROUND(max_price, 0) AS max_price,
    ROUND((total_transactions * 100.0 / SUM(total_transactions) OVER()), 2) AS percentage_market
FROM workspace.uk_housing.gold_price_category_analysis
WHERE town_city = 'LONDON'
ORDER BY avg_price DESC;

### Query 5 — Distritos de Londres
Top 15 distritos londinenses por precio medio. Refleja la heterogeneidad del mercado intraurbano.

In [0]:
-- ============================================
-- Identificar diferencias y evolución de precios dentro de los distritos de Londres
-- ============================================
SELECT 
    district,
    total_transactions,
    year,
    month,
    ROUND(avg_price, 0) AS avg_price,
    ROUND(min_price, 0) AS min_price,
    ROUND(max_price, 0) AS max_price
FROM workspace.uk_housing.gold_district_analysis
WHERE town_city = 'LONDON'
ORDER BY avg_price DESC
LIMIT 15;